# Stage 2 - MusicGen LoRA fine-tuning (Kaggle GPU)

Fine-tunes `facebook/musicgen-medium` with LoRA on the dataset from Stage 1.

**Before running:** Settings -> Accelerator -> **GPU T4 x2** (uses one GPU). Do **not** reinstall torch.

In [ ]:
import sys, os
from pathlib import Path
REPO_DIR = '/kaggle/working/metalcore'
if not Path(REPO_DIR).exists():
    !git clone https://github.com/YOUR_USERNAME/metalcore.git {REPO_DIR}
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Install stage deps (torch stays as Kaggle's preinstalled build).
!pip install -q -r requirements-music.txt

In [ ]:
DATASET_DIR = '/kaggle/working/dataset'          # from Stage 1 (or a Kaggle Dataset)
OUTPUT_DIR  = '/kaggle/working/outputs/music'
assert Path(DATASET_DIR, 'train.jsonl').exists(), 'Run Stage 1 first.'
print('Dataset:', DATASET_DIR, '\nOutput :', OUTPUT_DIR)

### Quick overfit sanity check (recommended)
Run a short training burst and confirm the loss decreases and a checkpoint is written.
Edit `configs/music_lora.yaml` (e.g. `max_steps: 50`) for a fast check, then set it back.

In [ ]:
# Full training (resume-safe). Re-run this exact cell after a session timeout.
!python -m music_training.cli train \
    --config configs/music_lora.yaml \
    --dataset {DATASET_DIR} \
    --output {OUTPUT_DIR} \
    --resume

In [ ]:
# Find the latest adapter and generate a sample.
ckpts = sorted(Path(OUTPUT_DIR, 'checkpoints').glob('step_*'))
adapter = str(ckpts[-1] / 'adapter'); print('Adapter:', adapter)
!python -m music_training.cli generate \
    --config configs/music_lora.yaml \
    --adapter {adapter} \
    --prompt "melodic metalcore chorus, soaring lead guitar, double bass drums" \
    --seconds 12 --num 2 \
    --output {OUTPUT_DIR}/generated

In [ ]:
import IPython.display as ipd
for wav in sorted(Path(OUTPUT_DIR, 'generated').glob('*.wav')):
    print(wav.name); ipd.display(ipd.Audio(str(wav)))